# Title: Notebook 2: Exploratory Data Analysis & Feature Space Representation

Mục tiêu cốt lõi của notebook này là khai phá các đặc trưng phân phối dữ liệu từ tập bình luận đã qua tiền xử lý, đồng thời tiến hành các kiểm định giả thuyết thống kê (Statistical Hypothesis Testing) để khám phá hành vi ngôn ngữ của người dùng. Trọng tâm của phân tích là sự so sánh đối chiếu có hệ thống giữa hai phương pháp biểu diễn không gian đặc trưng: Mô hình phân phối thưa (Sparse Representation - TF-IDF) và Mô hình nhúng ngữ nghĩa dày đặc (Dense Semantic Representation - Sentence Transformers). Thông qua việc đánh giá định lượng khả năng phân tách tuyến tính (Linear Separability) của cả hai không gian này, chúng ta sẽ thiết lập nền tảng lý thuyết vững chắc để ra quyết định kiến trúc mô hình học máy trong các giai đoạn tiếp theo.


## 0. Environment Setup & Data Loading

Tiến hành thiết lập môi trường bằng cách nhập (import) các thư viện phân tích dữ liệu, kiểm định thống kê và trực quan hóa chuyên dụng. Bộ dữ liệu `processed_labeled_reviews.csv` chứa các bình luận thương mại điện tử đã được chuẩn hóa, tokenize và gán nhãn ở Notebook 1 sẽ được nạp vào bộ nhớ để bắt đầu quá trình phân tích.


In [ ]:
# Data Manipulation and Statistical Analysis
import pandas as pd
import numpy as np
import scipy.stats as stats

# Data Visualization
import matplotlib.pyplot as plt
import seaborn as sns
from wordcloud import WordCloud

# Natural Language Processing & Feature Extraction
from sklearn.feature_extraction.text import TfidfVectorizer
from sentence_transformers import SentenceTransformer

# Machine Learning: Clustering, Dimensionality Reduction & Baseline Modeling
from sklearn.manifold import TSNE
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.linear_model import LogisticRegression, LinearSVC
from sklearn.model_selection import cross_val_score

# Configuration for plots
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette("husl")

# Load the processed and labeled dataset
# DATA_PATH = "../data/processed/processed_labeled_reviews.csv"
# df = pd.read_csv(DATA_PATH)
# df.info()


## 1. Class Imbalance Analysis

Phân tích phân phối nhãn (Class Distribution Analysis) là bước tiên quyết trong mọi pipeline học máy. Đối với văn bản đánh giá thương mại điện tử, dữ liệu thực tế hiếm khi đạt trạng thái cân bằng tự nhiên; người dùng thường có xu hướng để lại đánh giá tích cực (5 sao) nhiều hơn hẳn so với đánh giá tiêu cực hoặc trung lập. Sự mất cân bằng dữ liệu (Class Imbalance) ở mức độ nghiêm trọng sẽ tạo ra độ lệch (bias) lớn cho mô hình, khiến mô hình ưu tiên dự đoán lớp đa số và bỏ qua các đặc trưng vi mô của lớp thiểu số.


In [ ]:
# 1. Plot bar charts for sentiment/aspect label distribution
# 2. Compute and print the exact percentage distributions for each class

# Code Outline:
# sentiment_counts = df['sentiment_label'].value_counts()
# sentiment_pct = df['sentiment_label'].value_counts(normalize=True) * 100

# plt.figure(figsize=(8, 5))
# sns.barplot(x=sentiment_counts.index, y=sentiment_counts.values)
# plt.title("Sentiment Label Distribution")
# plt.ylabel("Number of Reviews")
# plt.show()

# print("Percentage Distribution:\n", sentiment_pct)


### Chiến lược xử lý (Mitigation Strategy)

*(Placeholder: Dựa trên biểu đồ phân phối, đánh giá mức độ mất cân bằng. Đề xuất các phương pháp giải quyết trong giai đoạn huấn luyện như sử dụng trọng số lớp (Class Weights), kỹ thuật tái lấy mẫu (SMOTE/Oversampling lớp thiểu số, Undersampling lớp đa số), hoặc sử dụng hàm mất mát Focal Loss nhằm ép mô hình tập trung học các mẫu khó phân loại.)*


## 2. Text Length Exploration & Statistical Testing

**Giả thuyết thống kê (Hypothesis):** *"Các bình luận tiêu cực thường có độ dài văn bản lớn hơn các bình luận tích cực, do những khách hàng không hài lòng có xu hướng miêu tả chi tiết lỗi sản phẩm và trải nghiệm tồi tệ của họ."*

Vì phân phối độ dài văn bản tự nhiên tuân theo phân phối lệch (như Log-Normal hoặc Poisson) và hiếm khi đạt chuẩn (Normal Distribution), việc sử dụng kiểm định tham số (như T-test) có thể dẫn đến sai lầm loại I (Type I error). Do đó, chúng ta sẽ áp dụng kiểm định phi tham số Mann-Whitney U Test để đánh giá sự khác biệt có ý nghĩa thống kê về trung vị độ dài giữa hai quần thể bình luận (Tích cực vs. Tiêu cực).


In [ ]:
# 1. Calculate word counts and character counts for 'cleaned_text'
# 2. Plot overlapping KDE/Histograms for lengths, separated by sentiment (Positive vs Negative)
# 3. Perform scipy.stats.mannwhitneyu test on the two distributions and print the p-value

# Code Outline:
# df['word_count'] = df['cleaned_text'].apply(lambda x: len(str(x).split()))

# plt.figure(figsize=(10, 6))
# sns.kdeplot(data=df[df['sentiment_label'] == 'tích cực'], x='word_count', label='Tích cực', fill=True)
# sns.kdeplot(data=df[df['sentiment_label'] == 'tiêu cực'], x='word_count', label='Tiêu cực', fill=True)
# plt.title("KDE Plot of Word Counts by Sentiment")
# plt.legend()
# plt.show()

# pos_lengths = df[df['sentiment_label'] == 'tích cực']['word_count'].dropna()
# neg_lengths = df[df['sentiment_label'] == 'tiêu cực']['word_count'].dropna()
# stat, p_value = stats.mannwhitneyu(neg_lengths, pos_lengths, alternative='greater')

# print(f"Mann-Whitney U Statistic: {stat}, P-value: {p_value:.5f}")


### Kết luận Thống kê (Statistical Conclusion)

*(Placeholder: Đưa ra nhận định bác bỏ hoặc chấp nhận giả thuyết không (Null Hypothesis) dựa trên giá trị p-value (< 0.05). Nếu kết quả có ý nghĩa thống kê, khẳng định rằng độ dài văn bản có thể đóng vai trò như một đặc trưng (feature) tiềm năng cho mô hình dự đoán.)*


## 3. Lexical Analysis & Zipf's Law

Việc thấu hiểu đặc tính từ vựng của văn bản thương mại điện tử là vô cùng quan trọng. Phương pháp phân tích phong phú từ vựng thông qua Tỷ lệ Loại/Thẻ (Type-Token Ratio - TTR) sẽ cho biết mức độ đa dạng ngôn từ của từng nhóm cảm xúc. Đồng thời, việc đối chiếu tần suất từ vựng với Định luật Zipf (Zipf's Law) giúp hệ thống đo lường được tỷ lệ "nhiễu" (noise) và tần suất xuất hiện của các từ hiếm (long-tail words). Nếu đồ thị tần suất lệch khỏi đường chuẩn Zipf, chứng tỏ văn bản chứa rất nhiều biệt ngữ, lỗi chính tả hoặc ngôn ngữ mạng đặc thù.


In [ ]:
# 1. Generate Word Clouds for Positive and Negative sentiments
# 2. Extract and plot Top 50 N-grams (n=1, 2, 3)
# 3. Calculate Type-Token Ratio (TTR) = (Unique words / Total words) for each class
# 4. Plot a log-log chart of term frequencies to check Zipf's Law

# Code Outline:
# from collections import Counter

# def plot_wordcloud(text_series, title):
#     wc = WordCloud(width=800, height=400, background_color='white').generate(" ".join(text_series.astype(str)))
#     plt.imshow(wc, interpolation='bilinear')
#     plt.title(title)
#     plt.axis('off')
#     plt.show()

# # Calculate TTR
# all_words = " ".join(df['cleaned_text'].astype(str)).split()
# unique_words = set(all_words)
# ttr = len(unique_words) / len(all_words) if len(all_words) > 0 else 0
# print(f"Global Type-Token Ratio (TTR): {ttr:.4f}")

# # Zipf's Law Verification (Log-Log Plot)
# word_counts = Counter(all_words)
# frequencies = sorted(list(word_counts.values()), reverse=True)
# ranks = np.arange(1, len(frequencies) + 1)

# plt.figure(figsize=(8, 6))
# plt.loglog(ranks, frequencies, marker=".")
# plt.title("Zipf's Law: Log-Log Plot of Term Frequencies")
# plt.xlabel("Log(Rank)")
# plt.ylabel("Log(Frequency)")
# plt.show()


### Đánh giá Ngữ vựng (Lexical Insights)

*(Placeholder: Trình bày các nhận xét sâu sắc về độ phong phú của từ vựng giữa bình luận tích cực và tiêu cực. Phân tích đồ thị Zipf để rút ra kết luận về sự cần thiết của các bộ lọc từ hiếm hoặc việc giới hạn kích thước từ điển (max_features) trong bước véc-tơ hóa tiếp theo nhằm tối ưu hóa bộ nhớ và giảm overfitting.)*


## 4. Sparse Feature Representation (TF-IDF)

Biểu diễn thưa (Sparse Representation) thông qua Term Frequency-Inverse Document Frequency (TF-IDF) là tiêu chuẩn kinh điển trong xử lý ngôn ngữ tự nhiên. Phương pháp này biến đổi văn bản thành không gian véc-tơ dựa trên tần suất xuất hiện và độ hiếm của N-gram. Trong bước này, chúng ta sẽ định lượng số chiều (dimensionality) của không gian tạo ra, tỷ lệ thưa thớt (Sparsity Ratio) của ma trận, và sử dụng t-SNE (t-Distributed Stochastic Neighbor Embedding) để trực quan hóa khả năng phân cụm tuyến tính của biểu diễn TF-IDF.


In [ ]:
# 1. Initialize TfidfVectorizer with ngram_range=(1,2) and fit_transform the text
# 2. Print the shape of the TF-IDF matrix and its Sparsity Ratio
# 3. Calculate Average Cosine Similarity (Intra-class vs Inter-class)
# 4. Apply t-SNE to reduce the matrix to 2D and plot a scatter plot colored by sentiment
# 5. Compute the Silhouette Score for the TF-IDF space

# Code Outline:
# tfidf = TfidfVectorizer(ngram_range=(1,2), max_features=5000)
# X_tfidf = tfidf.fit_transform(df['cleaned_text'].fillna(''))

# sparsity = (1.0 - (X_tfidf.nnz / (X_tfidf.shape[0] * X_tfidf.shape[1]))) * 100
# print(f"TF-IDF Matrix Shape: {X_tfidf.shape}")
# print(f"Sparsity Ratio: {sparsity:.2f}%")

# # Subset for t-SNE calculation to save memory/time
# tsne = TSNE(n_components=2, random_state=42)
# X_tfidf_tsne = tsne.fit_transform(X_tfidf.toarray()[:2000]) # Example on 2k samples

# # Plotting t-SNE
# sns.scatterplot(x=X_tfidf_tsne[:,0], y=X_tfidf_tsne[:,1], hue=df['sentiment_label'][:2000])
# plt.title("t-SNE Visualization of TF-IDF Space")
# plt.show()

# # Silhouette Score
# sil_tfidf = silhouette_score(X_tfidf.toarray()[:2000], df['sentiment_label'][:2000])
# print(f"TF-IDF Silhouette Score: {sil_tfidf:.4f}")


## 5. Dense Semantic Representation (Sentence Transformers)

Trái ngược với TF-IDF - phương pháp vô tình phá vỡ cấu trúc ngữ pháp và bỏ qua ngữ nghĩa sâu (semantic context), Biểu diễn ngữ nghĩa dày đặc (Dense Semantic Representation) thông qua các mô hình ngôn ngữ lớn như Sentence-BERT (`keepitreal/vietnamese-sbert`) mã hóa văn bản thành các véc-tơ không gian có số chiều cố định nhưng chứa đựng độ nén thông tin cực cao. Trong không gian này, các câu có ý nghĩa tương đồng sẽ có khoảng cách hình học gần nhau bất chấp việc chúng không dùng chung một từ vựng nào.

Nhiệm vụ của phần này là chứng minh bằng toán học (qua Silhouette Score) và trực quan hóa (qua t-SNE) để khẳng định sự vượt trội của không gian Dense trong việc biểu diễn đa nghĩa và sắc thái ngôn ngữ tiếng Việt so với mô hình Sparse truyền thống.


In [ ]:
# 1. Load keepitreal/vietnamese-sbert and encode the 'cleaned_text'
# 2. Apply t-SNE (2D) to the dense embeddings
# 3. Apply K-Means clustering (k=3) as a completely unsupervised test
# 4. Plot t-SNE scatter plot and compute Silhouette Score
# 5. Baseline Classification: Train LogisticRegression/LinearSVC (5-fold CV) on BOTH TF-IDF and SBERT to compare Macro F1-Scores

# Code Outline:
# sbert_model = SentenceTransformer('keepitreal/vietnamese-sbert')
# X_dense = sbert_model.encode(df['cleaned_text'].fillna('').tolist(), show_progress_bar=True)

# # t-SNE for Dense
# X_dense_tsne = tsne.fit_transform(X_dense[:2000])

# sns.scatterplot(x=X_dense_tsne[:,0], y=X_dense_tsne[:,1], hue=df['sentiment_label'][:2000])
# plt.title("t-SNE Visualization of SBERT Dense Space")
# plt.show()

# sil_dense = silhouette_score(X_dense[:2000], df['sentiment_label'][:2000])
# print(f"SBERT Silhouette Score: {sil_dense:.4f}")

# # Baseline Modeling Comparison
# clf = LinearSVC(random_state=42)
# tfidf_cv = cross_val_score(clf, X_tfidf, df['sentiment_label'], cv=5, scoring='f1_macro')
# sbert_cv = cross_val_score(clf, X_dense, df['sentiment_label'], cv=5, scoring='f1_macro')

# print(f"Baseline LinearSVC (TF-IDF) Macro F1: {np.mean(tfidf_cv):.4f}")
# print(f"Baseline LinearSVC (SBERT) Macro F1: {np.mean(sbert_cv):.4f}")


### Nhận xét Chiến lược (Strategic Conclusion)

*(Placeholder: So sánh định lượng và định tính giữa hai không gian biểu diễn dựa trên chỉ số Silhouette và Macro F1-Score từ mô hình cơ sở. Luận giải lý do tại sao Dense Embeddings ưu việt hơn trong việc nắm bắt sắc thái đồng nghĩa/trái nghĩa phức tạp (Ví dụ: "Hàng chất" và "Sản phẩm tốt" nằm gần nhau dù không chung từ vựng). Từ đó, xác lập cơ sở phương pháp luận để đưa Dense Embeddings vào làm đầu vào chính cho các kiến trúc mạng nơ-ron sâu hoặc phương pháp Contrastive Learning ở Notebook 3.)*
